# 01 · Descargar el vídeo y extraer el audio
### De YouTube a un modelo con personaje

El proyecto entero convierte unos vídeos en los que habla una persona en **un modelo de lenguaje que responde con su estilo**. Son seis pasos, uno por cuaderno, y cada uno deja sus archivos en una carpeta para el siguiente:

| | Cuaderno | Entra | Sale |
|---|---|---|---|
| 👉 **01** | **Descargar el vídeo y extraer el audio** | enlaces de YouTube | `datos/01_video/` · `datos/02_audio/*.wav` |
| 02 | Transcribir el audio | el audio | `datos/03_transcripcion/*.json` |
| 03 | Extraer al personaje | audio + transcripción | `datos/04_personaje/turnos.json` |
| 04 | Crear el archivo de entrenamiento | lo que dice el personaje | `datos/05_dataset/*.jsonl` |
| 05 | Fine-tuning (GPT, o modelo abierto en el 05b) | los `.jsonl` | `modelos/` |
| 06 | Usar el modelo | el modelo afinado | una conversación |

**En este cuaderno:** bajamos el vídeo con `yt-dlp` y nos quedamos solo con la voz, en un archivo `.wav`, usando `ffmpeg`.

> ⚖️ **Derechos.** Descarga solo vídeos que puedas usar: tuyos, con licencia Creative Commons o con permiso. El vídeo de ejemplo es una entrevista publicada por MediaLab UGR con licencia **CC BY**. Y recuerda que el modelo final imita la forma de hablar de una persona real: úsalo como ejercicio, nunca para hacerte pasar por ella.

In [ ]:
# === SETUP · ejecuta esta celda antes que ninguna otra ===
import os, sys, json, importlib

if "google.colab" in sys.modules:                  # en Colab, el proyecto vive en tu Drive
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/proyecto_personaje")
    get_ipython().system("pip -q install -r requirements.txt")

import config as cfg
importlib.reload(cfg)                               # recoge los cambios si acabas de editar config.py

print("Personaje:", cfg.PERSONAJE)
print("Proyecto :", cfg.RAIZ)

El personaje y la lista de vídeos están en **`config.py`**. Ábrelo: es corto y es el único archivo que hay que tocar para cambiar de personaje.

In [ ]:
print("Vídeos en config.py:")
for origen in cfg.VIDEOS:
    print("  ·", origen)

---
# 0 · ¿Están las herramientas?

| Herramienta | Qué hace | Cómo se instala |
|---|---|---|
| **yt-dlp** | Descarga vídeos de YouTube (y de mil sitios más) | `pip install yt-dlp` |
| **ffmpeg** | La navaja suiza del audio y el vídeo: convierte, corta, une, extrae | macOS: `brew install ffmpeg` · Windows: `winget install ffmpeg` · Linux: `sudo apt install ffmpeg` |

`ffmpeg` no es una librería de Python: es un programa. Lo llamaremos desde Python con `subprocess`.

In [ ]:
!pip install --upgrade youtube-dl

In [ ]:
import shutil, subprocess
import yt_dlp

for herramienta in ["ffmpeg", "ffprobe"]:
    ruta = shutil.which(herramienta)
    print(f"{herramienta:8}", "✅ " + ruta if ruta else "❌ no está instalado (mira la tabla de arriba)")
print(f"{'yt-dlp':8} ✅ versión {yt_dlp.version.__version__}")

---
# 1 · Mirar antes de descargar

`yt-dlp` puede pedir la **ficha** de un vídeo sin bajarlo: título, canal, duración, licencia y la lista de formatos disponibles. Conviene mirarla antes: así sabes cuánto dura y qué vas a descargar.

In [ ]:
def ficha_youtube(url):
    """Pide a YouTube los datos del vídeo SIN descargarlo."""
    with yt_dlp.YoutubeDL({"quiet": True, "no_warnings": True, "noplaylist": True}) as ydl:
        return ydl.extract_info(url, download=False)

info = ficha_youtube(cfg.VIDEOS[0])
for clave in ["id", "title", "channel", "duration", "upload_date", "license"]:
    print(f"{clave:12}", info.get(clave))

YouTube no guarda «un vídeo»: guarda **muchas versiones** del mismo (resoluciones distintas, y la imagen y el sonido por separado). Cada una es un *formato*.

In [ ]:
import pandas as pd

formatos = pd.DataFrame([{
    "id": f["format_id"], "ext": f["ext"], "resolución": f.get("resolution"),
    "vídeo": f.get("vcodec"), "audio": f.get("acodec"),
    "MB": round((f.get("filesize") or f.get("filesize_approx") or 0) / 1e6, 1),
} for f in info["formats"] if f.get("vcodec") != "none" or f.get("acodec") != "none"])
print(len(formatos), "formatos · estos son los que indican su tamaño:")
formatos[formatos["MB"] > 0]

Fíjate en las columnas `vídeo` y `audio`: casi todos los formatos tienen **solo imagen** (`audio = none`) o **solo sonido** (`vídeo = none`). Por eso hace falta `ffmpeg`: `yt-dlp` baja las dos pistas y `ffmpeg` las une.

---
# 2 · Descargar el vídeo

La opción clave es `format`, que es un pequeño lenguaje de selección:

| Trozo | Significa |
|---|---|
| `bv*[height<=480]` | la mejor pista de vídeo de 480p o menos |
| `+ba` | más la mejor pista de audio |
| `/b[height<=480]` | si eso no existe, el mejor formato «todo en uno» de 480p o menos |
| `/b` | y si tampoco, el mejor que haya |

Para lo que vamos a hacer la imagen da igual, así que 480p sobra y la descarga es rápida.

In [ ]:
from pathlib import Path

def descargar_video(url, altura_max=480):
    """Descarga el vídeo en datos/01_video y devuelve su ficha. Si ya está descargado, no lo repite."""
    opciones = {
        "format": f"bv*[height<={altura_max}]+ba/b[height<={altura_max}]/b",
        "merge_output_format": "mp4",                        # ffmpeg une imagen y sonido en un mp4
        "outtmpl": str(cfg.DIR_VIDEO / "%(id)s.%(ext)s"),    # nombre del archivo = id del vídeo
        "noplaylist": True, "quiet": True, "no_warnings": True, "noprogress": True,
    }
    with yt_dlp.YoutubeDL(opciones) as ydl:
        info = ydl.extract_info(url, download=True)
    archivo = next(p for p in cfg.DIR_VIDEO.glob(f"{info['id']}.*") if p.suffix not in {".part", ".ytdl"})
    return {"id": info["id"], "titulo": info["title"], "canal": info.get("channel"),
            "duracion_s": info.get("duration"),
            "licencia": info.get("license") or "licencia estándar de YouTube",
            "origen": url, "archivo": archivo.name}

ficha = descargar_video(cfg.VIDEOS[0])
ficha

In [ ]:
ruta_video = cfg.DIR_VIDEO / ficha["archivo"]
print(f"{ruta_video.name} · {ruta_video.stat().st_size / 1e6:.1f} MB")

from IPython.display import Video
Video(os.path.relpath(ruta_video), width=420)

> 🛟 **Si YouTube no te deja descargar** (error 403, «Sign in to confirm you're not a bot»…): 1) actualiza con `pip install -U yt-dlp`, que es lo que lo arregla casi siempre; 2) prueba desde otra red (en Colab falla a menudo porque YouTube bloquea sus servidores); 3) usa un archivo que ya tengas: en `config.py`, la lista `VIDEOS` admite también **rutas a archivos** de tu ordenador (mp4, mp3, m4a…). Más abajo está la función que los incorpora.

---
# 3 · Extraer el audio con `ffmpeg`

Del vídeo solo nos interesa la voz. Esta es la orden, pieza a pieza:

| Opción | Qué hace |
|---|---|
| `-i video.mp4` | archivo de entrada |
| `-vn` | **sin vídeo**: descarta la imagen |
| `-ac 1` | un solo canal (mono): la voz no necesita estéreo |
| `-ar 16000` | 16 000 muestras por segundo: lo que esperan Whisper y los modelos de voz |
| `-c:a pcm_s16le` | audio sin comprimir (WAV de 16 bits): cada paso posterior lo lee sin pérdidas |
| `-y` | sobrescribe si el archivo ya existe |

In [ ]:
def extraer_audio(ruta_video, ruta_audio, frecuencia=16000):
    """Deja en ruta_audio la pista de sonido del vídeo, en WAV mono de 16 kHz."""
    orden = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(ruta_video),
             "-vn", "-ac", "1", "-ar", str(frecuencia), "-c:a", "pcm_s16le", str(ruta_audio)]
    subprocess.run(orden, check=True)
    return ruta_audio

ruta_audio = extraer_audio(ruta_video, cfg.DIR_AUDIO / f"{ficha['id']}.wav")
print("Creado:", ruta_audio.relative_to(cfg.RAIZ))

---
# 4 · Comprobar el resultado

Nunca des por bueno un archivo sin abrirlo. Tres comprobaciones: **los números** (duración y frecuencia), **el dibujo** (¿hay sonido todo el rato?) y **el oído** (¿se entiende?).

In [ ]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

audio, fs = sf.read(ruta_audio, dtype="float32")
print(f"Muestras: {len(audio):,} · frecuencia: {fs} Hz · canales: {audio.ndim} · duración: {len(audio) / fs / 60:.1f} min")
print(f"Vídeo: {ruta_video.stat().st_size / 1e6:.1f} MB   ·   Audio WAV: {ruta_audio.stat().st_size / 1e6:.1f} MB")

# Volumen máximo de cada segundo: un dibujo rápido de dónde hay sonido y dónde silencio
segundos = len(audio) // fs
volumen = np.abs(audio[: segundos * fs]).reshape(segundos, fs).max(axis=1)
plt.figure(figsize=(11, 2.2))
plt.fill_between(np.arange(segundos) / 60, volumen, color="#4C72B0")
plt.xlabel("minuto"); plt.ylabel("volumen"); plt.title(ficha["titulo"]); plt.margins(x=0)
plt.show()

In [ ]:
from IPython.display import Audio

Audio(audio[: 20 * fs], rate=fs)     # los primeros 20 segundos

El audio es una lista de números: 16 000 por segundo. Eso es todo lo que va a «oír» el modelo de transcripción del cuaderno 02.

---
# 5 · Todos los vídeos de la lista

Lo mismo, en bucle, para cada elemento de `cfg.VIDEOS`. Al final se guarda **`datos/videos.json`**: el índice que usan los demás cuadernos para saber qué vídeos hay.

In [ ]:
def duracion(ruta):
    """Duración en segundos de cualquier archivo de audio o vídeo (se la preguntamos a ffprobe)."""
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(ruta)],
                       capture_output=True, text=True, check=True)
    return float(r.stdout)

def usar_archivo_local(ruta):
    """Incorpora al proyecto un archivo que ya tienes (vídeo o audio) como si viniera de YouTube."""
    ruta = Path(ruta).expanduser()
    destino = cfg.DIR_VIDEO / ruta.name
    if not destino.exists():
        shutil.copy(ruta, destino)
    return {"id": ruta.stem, "titulo": ruta.stem, "canal": None, "duracion_s": round(duracion(destino)),
            "licencia": "archivo propio", "origen": str(ruta), "archivo": destino.name}

fichas = []
for origen in cfg.VIDEOS:
    es_enlace = origen.startswith("http")
    ficha = descargar_video(origen) if es_enlace else usar_archivo_local(origen)
    wav = cfg.DIR_AUDIO / f"{ficha['id']}.wav"
    if not wav.exists():
        extraer_audio(cfg.DIR_VIDEO / ficha["archivo"], wav)
    fichas.append(ficha)
    print(f"✅ {ficha['id']:14} {ficha['duracion_s'] / 60:5.1f} min · {ficha['titulo']}")

cfg.FICHA_VIDEOS.write_text(json.dumps(fichas, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nÍndice guardado en {cfg.FICHA_VIDEOS.relative_to(cfg.RAIZ)} · total: {sum(f['duracion_s'] for f in fichas) / 60:.1f} minutos")

In [ ]:
pd.DataFrame(fichas)[["id", "titulo", "canal", "duracion_s", "licencia"]]

---
## Tu turno
1. **Tu personaje.** En `config.py` cambia `PERSONAJE`, `DESCRIPCION` y `VIDEOS` (dos o tres vídeos donde hable bastante). Vuelve a ejecutar el SETUP y la sección 5.
2. El WAV pesa mucho más que la pista de audio original de YouTube. ¿Por qué? Extrae el mismo audio en mp3 (`-c:a libmp3lame -b:a 64k`) y compara tamaños.
3. De un vídeo de dos horas solo te interesan diez minutos. Añade a `extraer_audio` dos parámetros, `inicio` y `fin`, que usen las opciones `-ss` y `-to` de `ffmpeg`.
4. **Reto:** si tu vídeo tiene música de fondo debajo de la voz, la transcripción empeora. Investiga [Demucs](https://github.com/adefossez/demucs) (`--two-stems vocals`), que separa la voz de la música.

In [ ]:
# Tu turno
...

---
**Siguiente paso → `02_transcribir_audio.ipynb`**: convertir ese WAV en texto, con el minuto y segundo de cada frase.